# CIS4020: Hall of Fame project, loaded data sample
Lahman Baseball Database, 2025 release (SABR). Data in `data/lahman_2025/`.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from data_prep import ensure_raw_data

DATA = ensure_raw_data()  # unzips data/raw/lahman_2025.zip on first run
pd.set_option('display.max_columns', 25)

def load(name):
    return pd.read_csv(DATA / f'{name}.csv', encoding='utf-8-sig', low_memory=False)

## Hall of Fame ballots (target)

In [2]:
hof = load('HallOfFame')
hof.head()

,playerID,yearid,votedBy,ballots,needed,votes,inducted,category,needed_note
0,aaronha01,1982,BBWAA,415.0,312.0,406.0,Y,Player,NaN
1,abbotji01,2005,BBWAA,516.0,387.0,13.0,N,Player,NaN
2,abreubo01,2020,BBWAA,397.0,298.0,22.0,N,Player,NaN
3,abreubo01,2021,BBWAA,401.0,301.0,35.0,N,Player,NaN
4,abreubo01,2022,BBWAA,394.0,296.0,34.0,N,Player,NaN


In [3]:
hof.shape

(6426, 9)

## Batting (career stats source)

In [4]:
batting = load('Batting')
batting.head()

,playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP
0,aardsda01,2004,1,SFN,NL,11,0,0,0,0,0,0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,aardsda01,2006,1,CHN,NL,45,2,0,0,0,0,0,0.0,0.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0
2,aardsda01,2007,1,CHA,AL,25,0,0,0,0,0,0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,aardsda01,2008,1,BOS,AL,47,1,0,0,0,0,0,0.0,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,0.0
4,aardsda01,2009,1,SEA,AL,73,0,0,0,0,0,0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
batting.shape

(128598, 22)

## Key tables used: file size, shape and key fields

In [6]:
key_fields = {
    'HallOfFame':    ('playerID, yearid, votedBy, votes, inducted, category', 'Target: ballot results and induction (Y/N)'),
    'Batting':       ('playerID, yearID, stint, G, AB, H, HR, RBI, SB, BB', 'Season batting stats -> career totals, OBP, SLG'),
    'Pitching':      ('playerID, yearID, W, L, SV, IPouts, ER, SO, ERA', 'Season pitching stats -> career totals, ERA, WHIP'),
    'People':        ('playerID, nameFirst, nameLast, birthYear, debut, finalGame', 'Player names and career span'),
    'AwardsPlayers': ('playerID, awardID, yearID', 'MVP, Cy Young, Gold Glove, Rookie of the Year counts'),
    'AllstarFull':   ('playerID, yearID, teamID', 'Number of All-Star selections'),
    'Appearances':   ('playerID, yearID, G_p, G_c, G_1b, G_2b, G_3b, G_ss, G_of, G_dh', 'Games by position -> primary position'),
}
rows = []
for t, (fields, use) in key_fields.items():
    df = load(t)
    rows.append({'table': t, 'file size (MB)': round((DATA / f'{t}.csv').stat().st_size / 1e6, 2),
                 'rows': len(df), 'columns': df.shape[1], 'key fields': fields, 'used for': use})
summary = pd.DataFrame(rows)
total = summary['file size (MB)'].sum()
print(f'Total for the {len(summary)} tables: {total:.1f} MB, {summary.rows.sum():,} rows')
with pd.option_context('display.max_colwidth', None):
    display(summary.style.hide(axis='index'))

Total for the 7 tables: 25.3 MB, 364,528 rows


table,file size (MB),rows,columns,key fields,used for
HallOfFame,0.310000,6426,9,"playerID, yearid, votedBy, votes, inducted, category",Target: ballot results and induction (Y/N)
Batting,8.000000,128598,22,"playerID, yearID, stint, G, AB, H, HR, RBI, SB, BB","Season batting stats -> career totals, OBP, SLG"
Pitching,5.110000,57630,30,"playerID, yearID, W, L, SV, IPouts, ER, SO, ERA","Season pitching stats -> career totals, ERA, WHIP"
People,3.220000,24270,25,"playerID, nameFirst, nameLast, birthYear, debut, finalGame",Player names and career span
AwardsPlayers,0.590000,12667,6,"playerID, awardID, yearID","MVP, Cy Young, Gold Glove, Rookie of the Year counts"
AllstarFull,0.260000,6425,8,"playerID, yearID, teamID",Number of All-Star selections
Appearances,7.830000,128512,21,"playerID, yearID, G_p, G_c, G_1b, G_2b, G_3b, G_ss, G_of, G_dh",Games by position -> primary position


## Target overview: player inductees by voting route

In [7]:
players = hof[hof['category'] == 'Player']
print('Distinct players ever on a ballot:', players['playerID'].nunique())
print('Players inducted:', players.loc[players['inducted'] == 'Y', 'playerID'].nunique())
players[players['inducted'] == 'Y']['votedBy'].value_counts().head(6)

Distinct players ever on a ballot: 1411
Players inducted: 281


votedBy
BBWAA                        137
Veterans                      77
Old Timers                    25
Negro League                  21
Veterans - Golden Age Era      4
Run Off                        3
Name: count, dtype: int64